## Logistic Regression Baseline - Results Summary

- **Model:** Logistic Regression + TF-IDF
- **Features:** 50,000 (bigrams, sublinear_tf=True)
- **Class weights:** balanced
- **Preprocessing:** lowercase, remove URLs, remove punctuation,
                   remove emojis, remove stop words, lemmatize

**Overall Test Results:**
- Macro F1:  0.69
- Accuracy:  0.70

**Per-Language Results:**
- Hausa achieved a Macro F1 of 0.71
- Igbo achieved a Macro F1 of 0.73  - the best
- Pidgin achieved a Macro F1 of 0.44  - the worst (Pidgin neutral F1: 0.13)
 Yoruba achieved a Macro F1 of 0.68

**Key Findings:**
- Pidgin language result is affected by the 72 neutral training samples
- Negative class is over-predicted across all languages
- Positive class has highest precision across all languages
- Random baseline would be 0.33 macro F1
- This model achieves 2x the random baseline — solid for TF-IDF

**This is the benchmark all subsequent models must beat.**

In [1]:
# import libraries

import numpy as np
import pandas as pd
import re
import nltk
import string
import pickle
from typing import List


from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline

In [2]:
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\osaze\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\osaze\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
# load the datasets
df_train = pd.read_csv('../01-data/02-processed/train_clean.csv')
df_test = pd.read_csv('../01-data/02-processed/test_clean.csv')
df_val = pd.read_csv('../01-data/02-processed/val_clean.csv')

In [4]:
# load datasets
languages = {
    'hausa' : 'hau',
    'igbo': 'ibo',
    'pidgin': 'pcm',
    'yoruba': 'yor'
}


df_all = []

for lang, folder in languages.items():
    lang =  pd.read_csv(f'../01-data/03-stopwords/{folder}.csv')
    df_all.append(lang)

stopword_df = pd.concat(df_all, ignore_index=True)

stopword = set(stopword_df['word'].to_list())


In [5]:
lemmatizer = WordNetLemmatizer()

def dataset_preprocessing(text):
    # lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r'http\S+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map
    "\U0001F1E0-\U0001F1FF"  # flags
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE
    )

    text = emoji_pattern.sub('', text)

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [word for word in tokens if word not in stopword]

    # Lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

In [6]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36569 entries, 0 to 36568
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   tweet         36569 non-null  object
 1   label         36569 non-null  object
 2   language      36569 non-null  object
 3   tweet_length  36569 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 1.1+ MB


In [7]:
# drop tweet length from train dataset
df_train.drop(columns=['tweet_length'], inplace=True)

df_list = [df_train, df_test, df_val]
for df in df_list:
    df['cleaned_tweet'] = df['tweet'].apply(dataset_preprocessing)
    df.drop(columns=['tweet'], inplace=True)
    df.rename(columns={'cleaned_tweet': 'tweet'}, inplace=True)

In [8]:
# split dataset for training, validation and testing
X_train = df_train['tweet']
y_train = df_train['label']
X_test = df_test['tweet']
y_test = df_test['label']
X_val = df_val['tweet']
y_val = df_val['label']

In [9]:
# train logistics regression model with max_feature as 20000 for initial check
log_reg = Pipeline([
    ('vectorizer', TfidfVectorizer(max_features=20000, ngram_range=(1,2), sublinear_tf=True)),
    ('model', LogisticRegression(max_iter=1000, class_weight="balanced"))
])

log_reg.fit(X_train, y_train)

predict = log_reg.predict(X_val)

f1_macro = f1_score(y_val, predict, average='macro')
print(f'f1_macro: {f1_macro}')

f1_macro: 0.7095689666388476


## Hyperparameter Tuning

In [10]:
def tune_base_model(features: list[int], X_train, y_train, X_val, y_val, **kwargs):
    
    scores = []

    for feature in features:
        log_reg = Pipeline([
        ('vectorizer', TfidfVectorizer(max_features=feature, 
                                       **kwargs)),
        ('model', LogisticRegression(max_iter=1000, class_weight="balanced"))
        ])

        log_reg.fit(X_train, y_train)

        predict = log_reg.predict(X_val)

        f1_macro = f1_score(y_val, predict, average='macro')

        accuracy = accuracy_score(y_val, predict)
        
        scores.append((feature, f1_macro, accuracy))

    return pd.DataFrame(scores, columns = ['max_feature', 'f1macro', 'accuracy'])

In [11]:
features = [10000, 20000, 50000, 60000, 70000]
ans = tune_base_model(features, X_train, y_train, X_val, y_val, ngram_range=(1,2), sublinear_tf=True)

In [12]:
ans

,max_feature,f1macro,accuracy
0,10000,0.700866,0.701013
1,20000,0.709569,0.709699
2,50000,0.716581,0.716673
3,60000,0.716692,0.716805
4,70000,0.717387,0.717463


*The max_feature with the best performance is the max_feature with value 70000, but we will go with max_feature of 50000 because for 20000 extra features and improvement of 0.0008, the extra memory and compute cost makes it not worth it. so we will use 50000 max_features value and retrain the dataset using that value and evaluate the performance using f1_macro*

In [13]:
log_reg = Pipeline([
    ('vectorizer', TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)),
    ('model', LogisticRegression(max_iter=1000, class_weight="balanced"))
])

log_reg.fit(X_train, y_train)

predict = log_reg.predict(X_test)

f1_macro = f1_score(y_test, predict, average='macro')
print(f'f1_macro: {f1_macro}')

f1_macro: 0.6938160380621619


In [14]:
# print the classification report
clf_report = classification_report(y_test, predict)
print(clf_report)

              precision    recall  f1-score   support

    negative       0.64      0.77      0.70      6009
     neutral       0.69      0.64      0.66      5457
    positive       0.77      0.67      0.72      6188

    accuracy                           0.70     17654
   macro avg       0.70      0.69      0.69     17654
weighted avg       0.70      0.70      0.69     17654



In [15]:
df_test['predicted'] = log_reg.predict(X_test)

for lang in ['hausa', 'igbo', 'pidgin', 'yoruba']:
    subset = df_test[df_test['language'] == lang]
    print(f"\n{'='*50}")
    print(f"Language: {lang.upper()}")
    print(f"{'='*50}")
    print(classification_report(subset['label'], subset['predicted']))


Language: HAUSA
              precision    recall  f1-score   support

    negative       0.65      0.77      0.71      1759
     neutral       0.68      0.60      0.64      1789
    positive       0.81      0.76      0.78      1755

    accuracy                           0.71      5303
   macro avg       0.71      0.71      0.71      5303
weighted avg       0.71      0.71      0.71      5303


Language: IGBO
              precision    recall  f1-score   support

    negative       0.70      0.65      0.67       943
     neutral       0.72      0.76      0.74      1621
    positive       0.78      0.76      0.77      1118

    accuracy                           0.73      3682
   macro avg       0.73      0.72      0.73      3682
weighted avg       0.73      0.73      0.73      3682


Language: PIDGIN
              precision    recall  f1-score   support

    negative       0.64      0.92      0.75      2326
     neutral       0.35      0.08      0.13       431
    positive       0.66 

In [17]:
# save the selected model
with open('../03-models/logreg_tfidf.pkl', 'wb') as f:
    pickle.dump(log_reg, f)
print("model saved to 03-models/logreg_tfidf.pkl")

model saved to 03-models/logreg_tfidf.pkl
